# 模型：量纲分析

本文内容源自博客文章 [Dimensional Analysis量纲分析入门](https://www.windtunnel.cn/posts/matlab/dynamics/dimensionalanalysis/)，并将其转化为 Python 实现的 Jupyter Notebook。

## 幂律 (Power Law)

一个核心定理是：**有单位的物理量，只能形成幂律关系。**

$$
y = k x_1^{n_1} x_2^{n_2} \cdots x_m^{n_m}
$$

其中，$k$ 是常数，$x_1, x_2, \cdots, x_m$ 是变量，$n_1, n_2, \cdots, n_m$ 是幂次。

### 简要证明

假设两个有单位的物理量 $y$ 和 $x$ 的关系为 $y = f(x)$。由于物理单位本身是比例关系，我们可以通过改变度量单位来引入一个缩放因子 $\alpha$。例如，从米到厘米，$\alpha=100$。

改变 $x$ 的单位， $x \rightarrow \alpha x$。相应地，$y$ 的单位也会改变 $y \rightarrow \beta y$。但物理规律本身不应随单位改变而改变，这意味着函数形式 $f$ 应该满足某种自洽性。一个与单位无关的公式可以表示为：

$$
\frac{f(\alpha x_1)}{f(\alpha x_2)} = \frac{f(x_1)}{f(x_2)}
$$

这个关系暗示了函数 $f$ 必须是幂律形式。对 $\alpha$ 求导并整理可以得到：

$$
\frac{x f'(x)}{f(x)} = \text{const}
$$

积分上式可得 $\ln f(x) = k \ln x + c$，最终导出：

$$
f(x) = C x^k
$$

这就是幂律关系。这个结论的本质在于，像 $\sin(l)$ 或 $e^l$ (其中 $l$ 是长度) 这样的表达式在物理上是无意义的，因为它们的泰勒展开会把不同量纲的项（如长度、面积、体积）加在一起。


## SI量纲

国际单位制（SI）定义了七个基本量纲：

*   **长度 (L)**, 单位：米 (m)
*   **质量 (M)**, 单位：千克 (kg)
*   **时间 (T)**, 单位：秒 (s)
*   **电流 (I)**, 单位：安培 (A)
*   **热力学温度 (Θ)**, 单位：开尔文 (K)
*   **物质的量 (N)**, 单位：摩尔 (mol)
*   **发光强度 (J)**, 单位：坎德拉 (cd)

任何物理量的量纲 `[y]` 都可以表示为这些基本量纲的幂次组合：

$$
[y] = L^a M^b T^c I^d \Theta^e N^f J^g
$$

根据幂律关系，一个物理量 $y$ 可以表示为其他物理参数 $a, b, c, \dots$ 的乘积：

$$
y = C a^\alpha b^\beta c^\gamma \cdots
$$

其量纲关系为：

$$
[y] = [a]^\alpha [b]^\beta [c]^\gamma \cdots
$$

将每个物理量的量纲用基本量纲表示，通过对比等式两边各基本量纲的幂次，就可以得到一组关于指数 $\alpha, \beta, \gamma, \dots$ 的线性方程组。


## 示例：单摆周期

我们来分析一个单摆的周期。假设单摆的角频率 $\omega$ 与摆长 $l$、摆锤质量 $m$、重力加速度 $g$ 有关。

![单摆](imgs/pendulum.png)

我们写出幂律关系：

$$
\omega = C l^\alpha m^\beta g^\gamma
$$

接下来进行量纲分析。首先定义各物理量的量纲：

*   $[\omega] = T^{-1}$ (角频率是 $2\pi/T_{period}$，量纲为时间的倒数)
*   $[l] = L$ (长度)
*   $[m] = M$ (质量)
*   $[g] = L T^{-2}$ (重力加速度)

代入量纲关系式：

$$
T^{-1} = (L)^\alpha (M)^\beta (L T^{-2})^\gamma = L^{\alpha+\gamma} M^\beta T^{-2\gamma}
$$

比较等式两边 `L`, `M`, `T` 的幂次，得到线性方程组：

*   L: $\alpha + \gamma = 0$
*   M: $\beta = 0$
*   T: $-2\gamma = -1$

我们可以用 `sympy` 来求解这个方程组。

In [ ]:
import sympy as sp

# 定义指数为符号
alpha, beta, gamma = sp.symbols('alpha beta gamma')

# 定义方程组
# L: alpha + gamma = 0
# M: beta = 0
# T: -2*gamma = -1
eq1 = sp.Eq(alpha + gamma, 0)
eq2 = sp.Eq(beta, 0)
eq3 = sp.Eq(-2 * gamma, -1)

# 求解方程组
solution = sp.solve((eq1, eq2, eq3), (alpha, beta, gamma))

print("求解得到的指数为:")
print(solution)

# 将解代入表达式
C = sp.Symbol('C')
l, m, g = sp.symbols('l m g')

omega_expr = C * l**solution[alpha] * m**solution[beta] * g**solution[gamma]

print("\n角频率 omega 的表达式为:")
sp.pprint(omega_expr)

求解结果为 $\alpha = -1/2, \beta = 0, \gamma = 1/2$。

这表明角频率 $\omega$ 与质量 $m$ 无关，最终表达式为：

$$
\omega = C l^{-1/2} g^{1/2} = C \sqrt{\frac{g}{l}}
$$

通过更详细的动力学分析可知，常数 $C=1$。单摆的周期 $T_{period} = 2\pi/\omega$，因此：

$$
T_{period} = 2\pi \sqrt{\frac{l}{g}}
$$

这与我们熟知的单摆周期公式完全一致。


## 进一步的讨论：无穷多解的情况

前面的例子很幸运，方程组有唯一解。但如果方程组的解不唯一（有无穷多解），情况会怎样呢？

考虑一个例子：一个质量为 $m$ 的物体以速度 $V$ 向上抛出，空气阻力与速度平方成正比，即 $F_d = k V^2$。我们想知道物体上升的最大高度 $h$ 与哪些因素有关。

写出幂律关系：

$$
h = C m^\alpha g^\beta k^\gamma V^\delta
$$

各物理量的量纲：
*   $[h] = L$
*   $[m] = M$
*   $[g] = L T^{-2}$
*   $[k] = [F_d]/[V^2] = (M L T^{-2}) / (L T^{-1})^2 = M L^{-1}$
*   $[V] = L T^{-1}$

代入量纲关系式：

$$
L^1 = (M)^\alpha (L T^{-2})^\beta (M L^{-1})^\gamma (L T^{-1})^\delta = L^{\beta-\gamma+\delta} M^{\alpha+\gamma} T^{-2\beta-\delta}
$$

得到方程组：
*   L: $\beta - \gamma + \delta = 1$
*   M: $\alpha + \gamma = 0$
*   T: $-2\beta - \delta = 0$

这个方程组有3个方程，4个未知数，因此有无穷多解。这预示着我们可以构建无量纲参数。

### 构造无量纲量 (Buckingham $\pi$ 定理)

当解不唯一时，我们可以将变量组合成无量纲的“$\pi$群”。我们的目标是找到一个无量纲组合 $\Pi$。

$$
\Pi = m^\alpha g^\beta k^\gamma V^\delta
$$

其量纲为 $L^0 M^0 T^0$。对应的方程组为：
*   L: $\beta - \gamma + \delta = 0$
*   M: $\alpha + \gamma = 0$
*   T: $-2\beta - \delta = 0$


In [ ]:
# 求解无量纲参数的方程组
# 这是一个齐次线性方程组，我们寻找其零空间
A = sp.Matrix([
    [0, 1, -1, 1],  # L
    [1, 0, 1, 0],   # M
    [0, -2, 0, -1]  # T
])

# 求解 A * x = 0
null_space = A.nullspace()

print("方程组 A*x=0 的零空间 (解空间) 为:")
print(null_space)

# 零空间给出了指数之间的关系
# 我们可以取其中一个基向量作为一组解，例如 null_space[0]
# alpha = 1, beta = 1, gamma = -1, delta = -2
# (为了得到正指数，可以取其相反数)
sol = -null_space[0]
print(f"\n取一组解: alpha={sol[0]}, beta={sol[1]}, gamma={sol[2]}, delta={sol[3]}")

# 构建无量纲参数 lambda
m, g, k, V = sp.symbols('m g k V')
lambda_pi = m**sol[0] * g**sol[1] * k**sol[2] * V**sol[3]

print("\n构造的无量纲参数 lambda 为:")
sp.pprint(lambda_pi)

我们找到了一个无量纲参数 $\lambda = \frac{mg}{kV^2}$。这个参数描述了重力与空气阻力的相对大小。

根据白金汉 $\pi$ 定理，原问题可以被简化。原来的关系 $h = f(m, g, k, V)$ 可以被写成一个只依赖于无量纲参数的函数关系。我们需要将 $h$ 也无量纲化。一个简单的无量纲高度是 $\frac{h}{m/k}$ (因为 $m/k$ 的量纲是 $L$)。

因此，原问题可以简化为：

$$
\frac{h}{m/k} = F\left(\frac{mg}{kV^2}\right)
$$

或者写成：

$$
h = \frac{m}{k} F(\lambda)
$$

通过这种方式，我们将一个依赖4个变量的复杂问题，简化为了一个只依赖1个无量纲变量的函数关系。这极大地简化了实验和理论分析。我们只需要研究函数 $F$ 的行为，而无需在4维参数空间中探索。


## 抛石问题分析与求解

现在我们来求解二维抛石问题的轨迹。其运动方程为：

$$
m\ddot{\vec{x}} = m\vec{g} - k|\dot{\vec{x}}|\dot{\vec{x}}
$$

展开为分量形式：

$$
m\ddot{x} = -k\sqrt{\dot{x}^2 + \dot{y}^2}\dot{x} \\
m\ddot{y} = -k\sqrt{\dot{x}^2 + \dot{y}^2}\dot{y} - mg
$$

初始条件：$t=0$ 时, $\vec{x}=(0,0)$, $\dot{\vec{x}}=(V\cos\theta, V\sin\theta)$。

我们使用前面得到的量纲特征来无量纲化方程。
*   特征长度: $L_{ref} = m/k$
*   特征速度: $V_{ref} = V$
*   特征时间: $T_{ref} = L_{ref}/V_{ref} = m/(kV)$

定义无量纲变量：

$$
X = \frac{x}{m/k}, \quad Y = \frac{y}{m/k}, \quad T = \frac{t}{m/(kV)}
$$

将这些关系代入原微分方程，经过链式法则求导和化简，可以得到无量纲的运动方程：

$$
\ddot{X} = -\sqrt{\dot{X}^2 + \dot{Y}^2}\dot{X} \\
\ddot{Y} = -\sqrt{\dot{X}^2 + \dot{Y}^2}\dot{Y} - \lambda
$$

其中 $\lambda = \frac{mg}{kV^2}$。

新的初始条件变为：$T=0$ 时, $X=0, Y=0, \dot{X}=\cos\theta, \dot{Y}=\sin\theta$。

整个问题从依赖5个参数 $(m, k, g, V, \theta)$ 简化为只依赖2个无量纲参数 $(\lambda, \theta)$。

下面我们使用 `scipy.integrate.solve_ivp` 来数值求解这个无量纲化的ODE。

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

def projectile_ode(T, U, lambda_val):
    """
    无量纲化的抛体运动ODE
    U = [X, Y, X_dot, Y_dot]
    """
    X, Y, X_dot, Y_dot = U
    speed = np.sqrt(X_dot**2 + Y_dot**2)
    
    X_ddot = -speed * X_dot
    Y_ddot = -speed * Y_dot - lambda_val
    
    return [X_dot, Y_dot, X_ddot, Y_ddot]

# 定义终止事件：当 Y 回到 0 时停止
def event_hit_ground(T, U, lambda_val):
    return U[1] # Y坐标
event_hit_ground.terminal = True # 触发时终止积分
event_hit_ground.direction = -1  # 从正到负穿越时触发

# 设置参数
lambda_val = 0.5  # mg / (kV^2)
theta_deg = 45    # 抛射角
theta_rad = np.deg2rad(theta_deg)

# 初始条件
X0, Y0 = 0, 0
X_dot0, Y_dot0 = np.cos(theta_rad), np.sin(theta_rad)
U0 = [X0, Y0, X_dot0, Y_dot0]

# 时间范围
T_span = [0, 10]

# 求解ODE
sol = solve_ivp(
    fun=projectile_ode,
    t_span=T_span,
    y0=U0,
    args=(lambda_val,),
    events=event_hit_ground,
    dense_output=True # 需要密集输出以绘制平滑曲线
)

# 绘制结果
T_plot = np.linspace(sol.t.min(), sol.t.max(), 500)
U_plot = sol.sol(T_plot)
X_plot = U_plot[0]
Y_plot = U_plot[1]

plt.figure(figsize=(10, 6))
plt.plot(X_plot, Y_plot, label=f'λ={lambda_val}, θ={theta_deg}°')
# 作为对比，绘制无阻力的情况 (lambda=0)
plt.plot(X_plot, X_plot * np.tan(theta_rad) - 0.5 * lambda_val * (X_plot / np.cos(theta_rad))**2, 'r--', label='No Drag Trajectory (Approx.)')


plt.title('Dimensionless Projectile Motion Trajectory')
plt.xlabel('Dimensionless Horizontal Distance X')
plt.ylabel('Dimensionless Vertical Distance Y')
plt.legend()
plt.grid(True)
plt.axis('equal')
plt.show()

print(f"求解状态: {sol.message}")
print(f"在无量纲时间 T = {sol.t.max():.2f} 时落地。")
print(f"无量纲射程 X_max = {sol.y[0, -1]:.2f}")
print(f"无量纲最大高度 Y_max = {np.max(sol.y[1]):.2f}")

## 无量纲化方法小结

量纲分析是一个强大的工具，它：

1.  **简化问题**：通过将多个物理参数组合成少数几个无量纲参数，显著减少了问题的复杂性。
2.  **指导实验**：实验设计可以专注于改变无量纲参数，而不是在多维的物理参数空间中盲目探索，从而大大节省成本和时间。
3.  **检验理论**：可以快速检查一个理论公式的量纲是否自洽。

通过将一个依赖5个变量的抛石问题简化为只依赖2个无量纲参数的问题，我们展示了量纲分析在理论和计算物理中的巨大价值。

## 无量纲分析方法在风洞模型中的应用

实际上，我们整个风洞总体气动设计的过程都在无量纲方法的框架下进行。

![管道流动阻力模型](./imgs/pipe.png)

对于图中的圆管，设其直径为 $D$，壁面粗糙度为 $\delta$（长度量纲），流动速度为 $V$，流体密度为 $\rho$，动力粘性系数为 $\mu$，壁面摩擦剪切应力为 $\tau_w$，目标是确定 $\tau_w$ 与这些参量之间的函数关系。

**Step 1** 列出问题涉及的全部参量（包括有量纲变量、无量纲变量和常数）并计算其个数。包括因变量问题设一共有 $n$ 个参量。对于这个问题，我们有 $6$ 个参量，即 $n=6$。我们的难题是需要确定关系

$$
\tau_w = f(V,\delta,\rho,\mu,D).
$$

**Step 2** 列出所有 $n$ 个参量的基本量纲（注意最多只有 $7$ 个！）。

| 参量 | $\tau_w$ | $V$ | $\delta$ | $\rho$ | $\mu$ | $D$ |
| --- | --- | --- | --- | --- | --- | --- |
| 基本量纲 | $m^1L^{-1}t^{-2}$ | $L^1t^{-1}$ | $L^1$ | $m^1L^{-3}$ | $m^1L^{-1}t^{-1}$ | $L^1$ |

**Step 3** 这个问题的基本量纲有 $L,m,t$ 共 $3$ 个。作为第一次猜测，取值等于问题中基本量纲的个数，即 $k=3$。这样我们就有 $k = n - j = 6 - 3 = 3$ 个无量纲 $\Pi$，这样就会有 $k = 3$ 个无量纲 $\Pi$。

**Step 4** 值是 $3$，我们要从 $6$ 个参量中选择 $3$ 个作为问题的重复参量，无量纲的攻角不选，因变量 $\tau_w$ 不选，粘性系数 $\mu$ 已经包含了所有的基本量纲（$L,m,t$），我们取 $D, V, \rho$ 作为重复参量。

3 个重复参量：$D, V, \rho$。

**Step 5** 把所有的无量纲量 $\Pi$ 列出来，必要时调整整理无量纲量 $\Pi$ 称为科学界已经命名的无量纲量。我们把 $\tau_w$ 与重复参量做一个乘积，但重复参量的幂次为待定。

$$
\Pi_1 = \tau_w V^a D^b \rho^c \rightarrow \Pi_1 = m^1 L t^{-2} (L t^{-1})^a (L^1)^b (m^1 L^{-3})^c = m^0 L^0 t^0.
$$

合并同类项，两边同一量的幂次必须相等，得到待定常数 $a=-2,\ b=-2,\ c=-1$。把确定的 $a$ 和 $b$ 带入到 $\Pi_1$，得到这个问题的 $\Pi$：

$$
\Pi_1 = \frac{\tau_w}{\rho V^2}.
$$

其中的 $\Pi_1$ 改写成传统的 Darcy 摩阻因子 $f_D$：

$$
\Pi_{1,\text{modified}} = \frac{8\tau_w}{\tfrac{1}{2} \rho V^2 A} = f.
$$

同样的，我们用空气粘性系数 $\mu$ 代替 $\tau_w$，重复计算就可以得到 $\Pi_2$：

$$
\Pi_2 = \frac{\mu}{\rho V L_c}.
$$

这个 $\Pi_2$ 就是 Reynolds 常数 $\operatorname{Re}$ 的倒数，可以改成传统表示

$$
\Pi_{2,\text{modified}} = \frac{\rho V L_c}{\mu} = \operatorname{Re}.
$$

同样可以得到第三个 $\Pi_3$，就是粗糙度 $\delta$ 与管直径的比值，传统称为粗糙度系数：

$$
\Pi_3 = \frac{\delta}{D}.
$$

**Step 6** 验证所有的无量纲量 $\Pi$，并写出问题的最终量纲关系。

$$
\Pi_{1,\text{modified}} = F(\Pi_{2,\text{modified}}, \Pi_3).
$$

得到摩阻因子为

$$
f = \frac{8\tau_w}{\rho V^2 A} = F(\operatorname{Re}, \delta/D).
$$

讨论：对于这个问题，我们没有建立问题的方程更没有求解，得到了摩阻因子与粗糙度系数的关系式，也很神奇。当然，量纲分析不能完全确定全部，待定常数 $F(\operatorname{Re}, \delta/D)$ 必须通过其他方法（如试验）确定。对于粗糙度系数 $\delta/D$ 为零的管流，在层流情况下 $f = 64/\operatorname{Re}$；对于湍流流况下的结果是 $f = 0.316\operatorname{Re}^{-1/4}$。对于速度 $V$ 很大的极限情况，可以认为 $\operatorname{Re} \to \infty$，对于这个问题 $F(\operatorname{Re}, \delta/D)$ 就只是粗糙度系数 $\delta/D$ 的函数。如果再完全光滑粗糙度系数为零，即

$$
f\big|_{V \to \infty,\, \delta/D \to 0} = f(0) = \text{constant} = C.
$$

这时摩阻因子为常数，即有剪切应力

$$
\tau_w = C \rho V^2.
$$